**za VZORCE testna zbirka**

In [1]:
import os
import struct
import random
import numpy as np
from PIL import Image

# 1. NASTAVITVE POTI
INPUT_DIR = r'F:\FRI_DIPLOMSKA\-delovna mapa-\DATASET\sken-OBRAZEC\vse'
OUTPUT_BIN = r'F:\FRI_DIPLOMSKA\-delovna mapa-\MODEL\bitmaps\zbirka_obrazec.bin'

LABELS_ORDERED = ['A', 'B', 'C', 'Č', 'D', 'E', 'F', 'G', 'H', 'I',
                  'J', 'K', 'L', 'M', 'N', 'O', 'P', 'R', 'S', 'Š',
                  'T', 'U', 'V', 'Z', 'Ž']
LABEL_TO_IDX = {l: i for i, l in enumerate(LABELS_ORDERED)}

# Slovar, kamor bomo grupirali slike po posameznih črkah
samples_by_char = {l: [] for l in LABELS_ORDERED}

# 2. BRANJE IN RAZVRŠČANJE SLIK
vse_datoteke = os.listdir(INPUT_DIR)

for f in vse_datoteke:
    if f.lower().endswith('.png'):
        # Predvidevamo format: Č_nekaj.png ali Č_oseba_št.png
        parts = f.split('_')
        crka = parts[0]
        
        if crka in samples_by_char:
            img_path = os.path.join(INPUT_DIR, f)
            try:
                # Naložimo sliko v sivi lestvici in jo spremenimo v 28x28
                img = Image.open(img_path).convert('L')
                if img.size != (28, 28):
                    img = img.resize((28, 28), Image.Resampling.LANCZOS)
                
                pixels = np.array(img, dtype=np.float32) / 255.0
                
                
                samples_by_char[crka].append(pixels.flatten())
            except Exception as e:
                print(f"Napaka pri branju datoteke {f}: {e}")

# 3. FILTRIRANJE IN IZBIRA NATANKO 9 VZORCEV ZA VSAKO ČRKO (PO VRSTI)
final_samples = []

# Ker gremo čez LABELS_ORDERED, bodo vzorci v končni datoteki zloženi točno po abecedi (A, A, A..., B, B, B...)
for crka in LABELS_ORDERED:
    pix_list = samples_by_char[crka]
    
    if len(pix_list) < 9:
        print(f"OPOZORILO: Za črko {crka} je na voljo le {len(pix_list)} slik (potrebno 9)! Vzamem vse.")
        selected = pix_list
    else:
        # Naključno izberemo 9 vzorcev iz nabora za to specifično črko
        # Uporabimo fiksno seme, da bo izbor ob vsakem zagonu enak
        random.seed(42)
        selected = random.sample(pix_list, 9)
        
    for pix in selected:
        final_samples.append({
            'label_idx': LABEL_TO_IDX[crka],
            'pixels': pix
        })

# Preverjanje skupnega števila
if len(final_samples) != 225:
    print(f"Skupno število vzorcev je {len(final_samples)} namesto 225. Preveri količino slik v mapi.")

# 4. IZVOZ V BINARNO DATOTEKO (Brez predhodnega mešanja!)
with open(OUTPUT_BIN, "wb") as fb:
    for sample in final_samples:
        # Format v C-ju: uint8_t label, uint16_t num_points (784), float[784] pixels
        data = struct.pack('<BH784f', sample['label_idx'], 784, *sample['pixels'])
        fb.write(data)

print(f"USPEH! Datoteka uspešno ustvarjena: {OUTPUT_BIN}")
print(f"Zapakiranih je {len(final_samples)} vzorcev, urejenih po abecedi (9x A, 9x B, 9x C ...).")

USPEH! Datoteka uspešno ustvarjena: F:\FRI_DIPLOMSKA\-delovna mapa-\MODEL\bitmaps\zbirka_obrazec.bin
Zapakiranih je 225 vzorcev, urejenih po abecedi (9x A, 9x B, 9x C ...).
